# CP1 · Notebook 01 — Setup y verificación

Este notebook verifica que tu entorno está listo para los siguientes 3 notebooks de CP1.

**Objetivo**: al final debe imprimir `✅ Setup OK — listo para 02`.

Si alguna celda falla, **detente** y revisa el `README.md` del CP1 antes de seguir.

## 1. Imports y versiones

Comprobamos que todas las librerías están instaladas con las versiones esperadas.

In [ ]:
import sys
import platform

print(f"Python:     {sys.version.split()[0]}")
print(f"Plataforma: {platform.system()} {platform.machine()}")

assert sys.version_info >= (3, 10), "Necesitas Python 3.10+. Actualiza tu entorno virtual."

In [ ]:
import numpy as np
import cv2
import torch
import torchvision
import matplotlib

print(f"numpy:       {np.__version__}")
print(f"opencv:      {cv2.__version__}")
print(f"torch:       {torch.__version__}    (cuda disponible: {torch.cuda.is_available()})")
print(f"torchvision: {torchvision.__version__}")
print(f"matplotlib:  {matplotlib.__version__}")

> **Nota**: si `cuda disponible: True` no pasa nada, pero **todo este CP está validado en CPU**. No hace falta GPU.

## 2. Verificación del dataset

Comprobamos que las imágenes están descargadas y organizadas correctamente.

In [ ]:
from pathlib import Path

# Ruta al dataset (relativa a la raíz del CP)
DATASET_DIR = Path("../datasets/lanes-subset")

# Conteos esperados según release cp1-v1 (14 imágenes, fuente Udacity CarND, MIT)
expected = {"easy": 5, "medium": 5, "hard": 4}
missing = []

for category, count_expected in expected.items():
    folder = DATASET_DIR / category
    if not folder.exists():
        missing.append(f"{category}/ no existe")
        continue
    images = list(folder.glob("*.png")) + list(folder.glob("*.jpg"))
    print(f"{category:8s}: {len(images):3d} imágenes (esperadas: {count_expected})")
    if len(images) < count_expected:
        missing.append(f"{category}/ tiene {len(images)}, faltan {count_expected - len(images)}")

if missing:
    print("\n❌ Dataset incompleto:")
    for m in missing:
        print(f"  - {m}")
    print("\nRevisa datasets/README.md y ejecuta `python scripts/download_assets.py`")
else:
    print("\n✅ Dataset OK")

## 3. Visualizar 3 imágenes muestra (una de cada dificultad)

Si esto se ve correctamente, OpenCV + matplotlib funcionan.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, category in zip(axes, ["easy", "medium", "hard"]):
    folder = DATASET_DIR / category
    sample = sorted(folder.glob("*.png"))[0] if folder.exists() else None
    if sample:
        # cv2 lee en BGR — convertimos a RGB para matplotlib
        img = cv2.imread(str(sample))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(f"{category}: {sample.name}")
        ax.axis("off")
    else:
        ax.text(0.5, 0.5, f"{category}/ vacío", ha="center", va="center")
        ax.axis("off")

plt.tight_layout()
plt.show()

## 4. Verificación de los pesos del modelo DL

El notebook `03_deep_learning.ipynb` usará un modelo pre-entrenado. Aquí solo comprobamos que los pesos están en disco; no cargamos el modelo todavía (lo haremos en 03).

In [ ]:
MODELS_DIR = Path("../models")
expected_weights = "cp1-lane-segmenter.onnx"
weight_path = MODELS_DIR / expected_weights

if weight_path.exists():
    size_mb = weight_path.stat().st_size / 1e6
    print(f"✅ Modelo encontrado: {weight_path.name}  ({size_mb:.1f} MB)")
else:
    print(f"⚠️  No encuentro {weight_path}")
    print("   Esto NO bloquea 02_clasico_canny_hough.ipynb,")
    print("   pero sí 03_deep_learning.ipynb.")
    print("   Ejecuta `python scripts/download_assets.py` antes de hacer 03.")

## 5. Cierre

Si todas las celdas anteriores no han fallado, estás listo. Ve a `02_clasico_canny_hough.ipynb`.

In [ ]:
if not missing:
    print("✅ Setup OK — listo para 02")
else:
    print("❌ Setup incompleto — revisa los warnings arriba")